# 002 · Generators

> **Syntax › Iterators & Generators** · Python 3.14

A **generator function** is a function that contains `yield`. Calling it does not run its body — it returns an iterator that runs the body step by step. If you have written a C# iterator method with `yield return`, you already know 90% of this.

**In this notebook you will learn:**

- how `yield` compares to C#'s `yield return`, and `return` to `yield break`
- lazy execution: why nothing runs until you iterate
- infinite generators and how to consume them safely
- delegating to another iterable with `yield from`
- generator state, and a brief look at `send()`
- chaining generators into processing pipelines

## C# mental model

| C# | Python |
|---|---|
| `IEnumerable<T> F() { yield return x; }` | `def f(): yield x` |
| `yield break;` | `return` (or simply falling off the end) |
| iterator method returns `IEnumerable<T>` | generator function returns a **generator** object (an iterator) |
| deferred execution — body runs on first `MoveNext()` | lazy — body runs on first `next()` |
| `foreach (var x in Inner()) yield return x;` | `yield from inner()` |
| return type `IEnumerable<int>` | type hint `Iterator[int]` / `Generator[int, None, None]` |

One difference up front: a C# iterator method returning `IEnumerable<T>` can be enumerated **many** times (each `foreach` restarts it). A Python generator object is an **iterator** — single-use.

## Your first generator

```csharp
// C#
static IEnumerable<int> CountUpTo(int n)
{
    for (var i = 1; i <= n; i++)
        yield return i;
}
```

In [ ]:
def count_up_to(n: int):
    for i in range(1, n + 1):
        yield i


gen = count_up_to(3)
print(gen)          # a generator object, not a list
print(list(gen))    # [1, 2, 3]

A generator is an ordinary iterator, so `next()`, `for`, `list()`, `sum()` all work:

In [ ]:
gen = count_up_to(3)
print(next(gen))     # 1
print(next(gen))     # 2
for n in gen:        # continues from where we left off
    print("loop:", n)

## Lazy execution — nothing runs until you iterate

Exactly like C#: calling the function only creates the generator. The body starts on the first `next()`, runs until the next `yield`, then **pauses** with all local variables intact.

In [ ]:
def noisy():
    print("  started")
    yield 1
    print("  resumed after 1")
    yield 2
    print("  finishing")


g = noisy()
print("generator created — body has not run yet")
print("got", next(g))
print("got", next(g))
print("got", next(g, "done"))

⚠️ **Gotcha:** argument validation inside a generator is also deferred — same trap as in C#, where you split an iterator into a validating wrapper and a private iterator method. In Python, the same pattern applies:

In [ ]:
def take_lazy(items, n):
    if n < 0:
        raise ValueError("n must be >= 0")   # only runs on first next()!
    for i, x in enumerate(items):
        if i >= n:
            return
        yield x


g = take_lazy([1, 2, 3], -1)         # no error yet...
print("created without error")
try:
    next(g)
except ValueError as e:
    print(f"{type(e).__name__}: {e}")   # ...error surfaces on first use

In [ ]:
# Fix: validate eagerly in a normal function, delegate to an inner generator
def take(items, n):
    if n < 0:
        raise ValueError("n must be >= 0")   # runs immediately

    def _gen():
        for i, x in enumerate(items):
            if i >= n:
                return
            yield x

    return _gen()


try:
    take([1, 2, 3], -1)
except ValueError as e:
    print(f"{type(e).__name__}: {e}")
print(list(take("abcdef", 3)))

## `return` ends the generator (≈ `yield break`)

A bare `return` stops the generator; the caller sees `StopIteration` (and a `for` loop simply ends).

```csharp
// C#
static IEnumerable<string> UntilBlank(IEnumerable<string> lines)
{
    foreach (var line in lines)
    {
        if (string.IsNullOrEmpty(line)) yield break;
        yield return line;
    }
}
```

In [ ]:
def until_blank(lines):
    for line in lines:
        if not line:
            return          # yield break
        yield line


print(list(until_blank(["alpha", "beta", "", "gamma"])))

💡 `return value` in a generator is legal — the value is attached to the `StopIteration` exception and becomes the result of a `yield from` expression. You will rarely need it outside coroutine-style code.

## Infinite generators

Because generators are lazy, they can describe **infinite** sequences — just never call `list()` on one. Take only what you need.

In [ ]:
def fibonacci():
    a, b = 0, 1
    while True:          # infinite — perfectly fine for a generator
        yield a
        a, b = b, a + b


fib = fibonacci()
print([next(fib) for _ in range(10)])

In [ ]:
from itertools import islice   # islice ≈ LINQ Take / Skip

print(list(islice(fibonacci(), 10, 15)))   # Skip(10).Take(5)

for n in fibonacci():
    if n > 100:
        break
    print(n, end=" ")

## `yield from` — delegating to another iterable

```csharp
// C# has no shorthand — you write the loop yourself
static IEnumerable<int> Flatten(IEnumerable<IEnumerable<int>> groups)
{
    foreach (var g in groups)
        foreach (var x in g)
            yield return x;
}
```

`yield from iterable` yields every item of `iterable` in turn:

In [ ]:
def flatten(groups):
    for g in groups:
        yield from g          # instead of: for x in g: yield x


print(list(flatten([[1, 2], (3,), range(4, 6)])))

It shines in recursive generators, such as walking a tree:

In [ ]:
tree = {"value": 1, "children": [
    {"value": 2, "children": [{"value": 4, "children": []}]},
    {"value": 3, "children": []},
]}


def walk(node):
    yield node["value"]
    for child in node["children"]:
        yield from walk(child)


print(list(walk(tree)))   # depth-first: [1, 2, 4, 3]

## Generator state

A generator object remembers where it paused. `inspect.getgeneratorstate()` shows its lifecycle: created → suspended → closed.

In [ ]:
import inspect

g = count_up_to(2)
print(inspect.getgeneratorstate(g))   # GEN_CREATED
next(g)
print(inspect.getgeneratorstate(g))   # GEN_SUSPENDED
list(g)                               # run to the end
print(inspect.getgeneratorstate(g))   # GEN_CLOSED

`close()` stops a suspended generator early — it raises `GeneratorExit` *inside* the generator so `finally` blocks run. This is similar to how disposing a C# iterator runs its `finally` blocks when `foreach` exits early.

In [ ]:
def reader():
    try:
        for i in range(100):
            yield i
    finally:
        print("  cleanup ran")   # like a finally in a C# iterator on Dispose()


r = reader()
print(next(r), next(r))
r.close()
print(inspect.getgeneratorstate(r))

## `send()` — passing values in (briefly)

Generators are two-way: `yield` is also an **expression** whose value is whatever the caller passes to `gen.send(value)`. C# iterators have no equivalent. This is the historical foundation of Python coroutines; in modern code you will mostly use `async`/`await` instead *(see Async › 001 · Async and Await)*.

In [ ]:
def running_average():
    total, count = 0.0, 0
    average = None
    while True:
        value = yield average     # pause; receive the sent value
        total += value
        count += 1
        average = total / count


avg = running_average()
next(avg)                 # "prime" it: run to the first yield
print(avg.send(10))       # 10.0
print(avg.send(20))       # 15.0
print(avg.send(60))       # 30.0

## Generators as pipelines

Small generators that each take an iterable and yield results can be chained like LINQ operators. Each item flows through the whole pipeline one at a time, and nothing is materialised in memory.

```csharp
// C#
var total = lines
    .Select(l => l.Trim())
    .Where(l => l.Length > 0 && !l.StartsWith('#'))
    .Select(int.Parse)
    .Sum();
```

In [ ]:
raw_lines = ["  10", "# comment", "", "20 ", "  5  "]


def stripped(lines):
    for line in lines:
        yield line.strip()


def meaningful(lines):
    for line in lines:
        if line and not line.startswith("#"):
            yield line


def as_ints(lines):
    for line in lines:
        yield int(line)


pipeline = as_ints(meaningful(stripped(raw_lines)))   # nothing has run yet
print(sum(pipeline))                                  # 35

For one-line steps like these, **generator expressions** are even shorter *(see Iterators & Generators › 003 · Generator Expressions and Lazy Evaluation)*.

## Type hints for generators

Annotate a simple generator as returning `Iterator[T]` (≈ `IEnumerator<T>`) or `Iterable[T]`. The full form `Generator[Yield, Send, Return]` is only needed when you use `send()` or `return value`.

In [ ]:
from collections.abc import Generator, Iterator


def evens(limit: int) -> Iterator[int]:
    yield from range(0, limit, 2)


def averager() -> Generator[float | None, float, None]:
    total, count = 0.0, 0
    while True:
        value = yield (total / count if count else None)
        total, count = total + value, count + 1


print(list(evens(10)))

## Summary

| Concept | C# | Python |
|---|---|---|
| Produce a value | `yield return x;` | `yield x` |
| Stop early | `yield break;` | `return` |
| Delegate to another sequence | nested `foreach` + `yield return` | `yield from other` |
| Execution | deferred until `MoveNext()` | lazy until `next()` |
| Re-enumerate | calling `foreach` again restarts | generator object is single-use — call the function again |
| Early-exit cleanup | `Dispose()` runs `finally` | `close()` runs `finally` |
| Two-way communication | — | `gen.send(value)` |
| Return type | `IEnumerable<T>` | `Iterator[T]` / `Generator[Y, S, R]` |

## Exercises

1. Write a generator `chunks(items, size)` that yields lists of up to `size` items (`chunks(range(5), 2)` → `[0, 1], [2, 3], [4]`).
2. Write an infinite generator `cycle_colors()` that yields `"red", "green", "blue", "red", ...`. Print the first 7 values with `islice`.
3. Write a recursive generator `flatten_deep(x)` that flattens arbitrarily nested lists: `[1, [2, [3, [4]]], 5]` → `1 2 3 4 5`.

In [ ]:
# Your solutions here

<details>
<summary>Show solutions</summary>

```python
from itertools import islice

# 1
def chunks(items, size):
    batch = []
    for x in items:
        batch.append(x)
        if len(batch) == size:
            yield batch
            batch = []
    if batch:
        yield batch

print(list(chunks(range(5), 2)))   # [[0, 1], [2, 3], [4]]

# 2
def cycle_colors():
    while True:
        yield from ("red", "green", "blue")

print(list(islice(cycle_colors(), 7)))

# 3
def flatten_deep(x):
    for item in x:
        if isinstance(item, list):
            yield from flatten_deep(item)
        else:
            yield item

print(list(flatten_deep([1, [2, [3, [4]]], 5])))
```

</details>